In [0]:
%sql
-- Sentiment + message recency per lead_id
-- Data originates from student_sentiment source, already joined to leads by CRM
-- and stored in leads_processed_custom. clean \N → NULL and exclude C082U471YMU.
CREATE OR REPLACE TABLE crm_ingestion.gold.sentiment AS (
SELECT
    leads_processed_id AS lead_id,
    NULLIF(SENTIMENTS_LAST_30_DAYS, '\\N') AS sentiment,
    TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT AS BIGINT) AS days_since_last_message_from_client,
    TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER AS BIGINT) AS days_since_last_message_from_team_member
FROM (
    SELECT
        leads_processed_id,
        Channel_ID,
        SENTIMENTS_LAST_30_DAYS,
        DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        ROW_NUMBER() OVER(PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
    FROM
        crm_ingestion.silver.leads_processed_custom
    WHERE
        Channel_ID IS NOT NULL
        AND Channel_ID != ''
        AND Channel_ID != 'C082U471YMU'  -- excluded channel (spec 5.1)
)
WHERE rn = 1
)